In [1]:
# =============================================================================
# 01_feature_taxonomy_and_actionability_audit.ipynb
# =============================================================================
# Purpose:
#   Establish the actionability taxonomy of the 30 SHAP-selected features
#   (owner-controllable "sales" variables vs. non-controllable "population"
#   context variables) and audit, at the population level, how often the
#   ORIGINAL (unconstrained) DiCE counterfactuals recommend changes to
#   non-controllable features. This motivates the actionability-constrained
#   reformulation studied in the companion notebooks.
#
# Inputs:
#   ../data/modeling_dataset_top30.csv          (original feature values)
#   ../data/selected_features.pkl               (30 feature names)
#   ../data/top30_features_list.csv             (mean |SHAP| ranking)
#   ../data/original_outputs/cf_all_experiments.csv  (unconstrained CFs, 2/4/6/8)
#
# Outputs (results/):
#   tables/t1_feature_taxonomy.csv
#   tables/t2_actionability_audit.csv
#   figures/f1_shap_by_actionability.(png|pdf)
#   figures/f2_immutable_change_share.(png|pdf)
#   figures/f3_contaminated_cf_rate.(png|pdf)
# =============================================================================

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# Global plotting style: grayscale, no titles/captions, 600 dpi, png + pdf
# ----------------------------------------------------------------------------
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.size": 11, "axes.edgecolor": "black", "axes.linewidth": 0.8,
    "savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42, "ps.fonttype": 42,
})
GRAY_DARK, GRAY_MID, GRAY_LIGHT = "0.20", "0.45", "0.75"
FIG_DIR = Path("../results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = Path("../results/tables");  TAB_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    """Save a figure as both PNG and PDF at 600 dpi without a caption/title."""
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"  saved figure: {name}.png / {name}.pdf")

# ----------------------------------------------------------------------------
# PART 1: Load data and define the actionability taxonomy
# ----------------------------------------------------------------------------
print("=" * 78); print("PART 1: FEATURE ACTIONABILITY TAXONOMY"); print("=" * 78)

STORE_ID = "store_id"
features = joblib.load("../data/selected_features.pkl")
df_orig = pd.read_csv("../data/modeling_dataset_top30.csv")
shap_rank = pd.read_csv("../data/top30_features_list.csv")

# An individual restaurant owner can influence its own realised sales mix
# (by targeting, menu, pricing, promotions, operating hours) but cannot change
# the resident / working / floating population of the surrounding district.
ACTIONABLE = [f for f in features if "Sales" in f]       # owner-controllable
IMMUTABLE  = [f for f in features if "Population" in f]   # district context
assert set(ACTIONABLE) | set(IMMUTABLE) == set(features), "unclassified feature!"

taxonomy = (shap_rank.assign(
        Actionability=lambda d: np.where(d["Feature"].isin(ACTIONABLE),
                                         "actionable", "immutable"),
        Group=lambda d: np.where(d["Feature"].str.contains("Sales"), "Sales",
                        np.where(d["Feature"].str.contains("Floating"), "Floating pop.",
                        np.where(d["Feature"].str.contains("Resident"), "Resident pop.",
                                 "Working pop."))))
      [["Rank", "Feature", "Mean_Abs_SHAP", "Group", "Actionability"]])
taxonomy.to_csv(TAB_DIR / "t1_feature_taxonomy.csv", index=False)

print(f"\nTotal features : {len(features)}")
print(f"  Actionable (sales)     : {len(ACTIONABLE)}")
print(f"  Immutable  (population): {len(IMMUTABLE)}")
imm_in_top10 = taxonomy.query("Rank<=10 and Actionability=='immutable'").shape[0]
print(f"\nImmutable features among the 10 most important (|SHAP|): {imm_in_top10}/10")
print("\nTaxonomy (saved to t1_feature_taxonomy.csv):")
print(taxonomy.to_string(index=False))

# ----------------------------------------------------------------------------
# PART 2: Population-level actionability audit of unconstrained counterfactuals
# ----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("PART 2: ACTIONABILITY AUDIT OF UNCONSTRAINED DiCE COUNTERFACTUALS")
print("=" * 78)

cf = pd.read_csv("../data/original_outputs/cf_all_experiments.csv")
cf = cf[cf["success"] == True].copy()
EXP2NCF = {1: 2, 2: 4, 3: 6, 4: 8}
EPS = 1e-6

# Align each counterfactual row with its store's original feature vector.
orig_feat = df_orig.set_index(STORE_ID)[features]
orig_aligned = orig_feat.reindex(cf[STORE_ID].values).reset_index(drop=True)
cf_feat = cf[features].reset_index(drop=True)

changed = (cf_feat.values - orig_aligned.values)
changed_mask = np.abs(changed) > EPS                     # (n_cf, 30) boolean
imm_idx = [features.index(f) for f in IMMUTABLE]
act_idx = [features.index(f) for f in ACTIONABLE]

n_changes = changed_mask.sum(axis=1)                     # changes per CF
n_imm = changed_mask[:, imm_idx].sum(axis=1)             # immutable changes per CF
n_act = changed_mask[:, act_idx].sum(axis=1)             # actionable changes per CF
has_imm = n_imm > 0                                      # CF touches an immutable feature

audit_df = pd.DataFrame({
    "experiment_id": cf["experiment_id"].values,
    "n_changes": n_changes, "n_imm": n_imm, "n_act": n_act, "has_imm": has_imm,
})
audit_df = audit_df[audit_df["n_changes"] > 0]           # ignore degenerate CFs

rows = []
for exp in sorted(audit_df["experiment_id"].unique()):
    s = audit_df[audit_df["experiment_id"] == exp]
    rows.append({
        "Num_CFs": EXP2NCF[exp],
        "N_counterfactuals": len(s),
        "Mean_changes_per_CF": s["n_changes"].mean(),
        "Immutable_change_share": s["n_imm"].sum() / s["n_changes"].sum(),
        "Actionable_change_share": s["n_act"].sum() / s["n_changes"].sum(),
        "Contaminated_CF_rate": s["has_imm"].mean(),  # >=1 non-actionable change
    })
audit_tab = pd.DataFrame(rows).sort_values("Num_CFs").reset_index(drop=True)
audit_tab.to_csv(TAB_DIR / "t2_actionability_audit.csv", index=False)

print("\nPer-setting audit (saved to t2_actionability_audit.csv):")
print(audit_tab.round(4).to_string(index=False))
overall_imm = audit_df["n_imm"].sum() / audit_df["n_changes"].sum()
overall_cont = audit_df["has_imm"].mean()
print(f"\nPooled across all settings:")
print(f"  Share of feature changes on immutable (non-controllable) features: {overall_imm:.1%}")
print(f"  Counterfactuals containing >=1 non-actionable change             : {overall_cont:.1%}")

# ----------------------------------------------------------------------------
# PART 3: Figures
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 3: FIGURES"); print("=" * 78)

# F1: mean|SHAP| of the top-30, shaded by actionability (dark=actionable).
t = taxonomy.sort_values("Mean_Abs_SHAP")
colors = np.where(t["Actionability"].values == "actionable", GRAY_DARK, GRAY_LIGHT)
fig, ax = plt.subplots(figsize=(7.2, 8.4))
ax.barh(t["Feature"], t["Mean_Abs_SHAP"], color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("mean(|SHAP value|)")
ax.set_ylabel("")
handles = [plt.Rectangle((0, 0), 1, 1, facecolor=GRAY_DARK, edgecolor="black"),
           plt.Rectangle((0, 0), 1, 1, facecolor=GRAY_LIGHT, edgecolor="black")]
ax.legend(handles, ["Actionable (sales)", "Immutable (population)"],
          loc="lower right", frameon=True)
ax.tick_params(axis="y", labelsize=8)
save_fig(fig, "f1_shap_by_actionability")

# F2: distribution of the per-CF immutable-change share (optimal 2-CF setting).
share = (audit_df.loc[audit_df["experiment_id"] == 1]
         .eval("n_imm / n_changes"))
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.hist(share, bins=np.linspace(0, 1, 11), color=GRAY_MID,
        edgecolor="black", linewidth=0.6)
ax.set_xlabel("Share of a counterfactual's changes that are non-actionable")
ax.set_ylabel("Number of counterfactuals")
save_fig(fig, "f2_immutable_change_share")

# F3: contaminated-CF rate and immutable-change share by number of CFs.
fig, ax = plt.subplots(figsize=(6.4, 4.2))
x = np.arange(len(audit_tab)); w = 0.38
ax.bar(x - w/2, audit_tab["Immutable_change_share"], width=w,
       color=GRAY_DARK, edgecolor="black", label="Immutable change share")
ax.bar(x + w/2, audit_tab["Contaminated_CF_rate"], width=w,
       color=GRAY_LIGHT, edgecolor="black", label="Contaminated-CF rate")
ax.set_xticks(x); ax.set_xticklabels([f"{n} CFs" for n in audit_tab["Num_CFs"]])
ax.set_ylabel("Proportion"); ax.set_ylim(0, max(0.4, audit_tab[["Immutable_change_share","Contaminated_CF_rate"]].values.max()*1.25))
ax.legend(frameon=True, loc="upper right")
save_fig(fig, "f3_contaminated_cf_rate")

print("\n[NB1 DONE] taxonomy + actionability audit complete.")


PART 1: FEATURE ACTIONABILITY TAXONOMY

Total features : 30
  Actionable (sales)     : 19
  Immutable  (population): 11

Immutable features among the 10 most important (|SHAP|): 3/10

Taxonomy (saved to t1_feature_taxonomy.csv):
 Rank                               Feature  Mean_Abs_SHAP         Group Actionability
    1                   Age_40_Sales_Amount       0.117821         Sales    actionable
    2                   Age_50_Sales_Amount       0.036857         Sales    actionable
    3                  Tuesday_Sales_Amount       0.034133         Sales    actionable
    4           TimeSlot_21-24_Sales_Amount       0.024386         Sales    actionable
    5                   Age_20_Sales_Amount       0.022578         Sales    actionable
    6                    Age_50_Sales_Count       0.020338         Sales    actionable
    7           TimeSlot_17-21_Sales_Amount       0.018627         Sales    actionable
    8      Female_Age_10_Working_Population       0.016667  Working pop.   


Per-setting audit (saved to t2_actionability_audit.csv):
 Num_CFs  N_counterfactuals  Mean_changes_per_CF  Immutable_change_share  Actionable_change_share  Contaminated_CF_rate
       2               8358               1.6890                  0.2603                   0.7397                0.2563
       4              16716               1.9904                  0.2173                   0.7827                0.2343
       6              25074               2.0260                  0.2542                   0.7458                0.3131
       8              33432               2.0670                  0.2477                   0.7523                0.2918

Pooled across all settings:
  Share of feature changes on immutable (non-controllable) features: 24.5%
  Counterfactuals containing >=1 non-actionable change             : 28.3%

PART 3: FIGURES


  saved figure: f1_shap_by_actionability.png / f1_shap_by_actionability.pdf


  saved figure: f2_immutable_change_share.png / f2_immutable_change_share.pdf


  saved figure: f3_contaminated_cf_rate.png / f3_contaminated_cf_rate.pdf

[NB1 DONE] taxonomy + actionability audit complete.
